# 🏗️ 02 — Final Training Pipeline

**Fraud Detection Capstone — Sprint 4: Final Model, Deployment & Portfolio**

## 🎯 Goal
A single, clean, reproducible pipeline that goes from raw `creditcard.csv` straight to the final saved model artifact — everything from Sprints 1–3 consolidated into one script, ready to hand to someone else (or a deployment app) with no manual steps in between.


In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---- Config (final choices from Sprints 1-3) ----
DATA_PATH = Path("data/creditcard.csv")
MODEL_OUT = Path("models/final_model.keras")
SCALER_OUT = Path("models/final_scaler.joblib")
FINAL_THRESHOLD = 0.93
DROPOUT = (0.5, 0.4, 0.3)
LEARNING_RATE = 0.0005
BATCH_SIZE = 64
EPOCHS = 60

print("Config loaded.")


I0000 00:00:1788704569.686758    9940 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1788704569.734863    9940 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1788704571.509229    9940 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Config loaded.


## 1. Load & Clean

In [2]:
df = pd.read_csv(DATA_PATH)
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Loaded {n_before:,} rows, dropped {n_before - len(df):,} duplicates -> {len(df):,} rows")
print(f"Fraud rate: {df['Class'].mean()*100:.4f}%")


Loaded 5,009 rows, dropped 0 duplicates -> 5,009 rows
Fraud rate: 0.1797%


## 2. Split, Scale, Engineer Features

In [3]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

def add_engineered_features(X):
    X = X.copy()
    X["Hour"] = (X["Time"] // 3600) % 24
    X["Amount_log"] = np.log1p(X["Amount"])  # raw Amount, always >= 0 -- must run BEFORE scaling
    return X

# Feature engineer on RAW values first, then scale -- log1p needs the real
# dollar amount, not the standardized (possibly negative) value.
X_train = add_engineered_features(X_train)
X_val = add_engineered_features(X_val)
X_test = add_engineered_features(X_test)

scale_cols = ["Time", "Amount"]
scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val[scale_cols] = scaler.transform(X_val[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

Path("models").mkdir(exist_ok=True)
joblib.dump({"scaler": scaler, "scale_cols": scale_cols, "feature_order": list(X_train.columns)}, SCALER_OUT)
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"NaNs check: {X_train.isnull().sum().sum()}")
print(f"Saved preprocessing artifact -> {SCALER_OUT}")


Train: (3506, 32) | Val: (751, 32) | Test: (752, 32)
NaNs check: 0
Saved preprocessing artifact -> models/final_scaler.joblib


## 3. Build & Train the Final Architecture

In [4]:
def build_final_model(n_features):
    return Sequential([
        Dense(64, activation="relu", input_shape=(n_features,)),
        BatchNormalization(), Dropout(DROPOUT[0]),
        Dense(32, activation="relu"),
        BatchNormalization(), Dropout(DROPOUT[1]),
        Dense(16, activation="relu"),
        BatchNormalization(), Dropout(DROPOUT[2]),
        Dense(1, activation="sigmoid"),
    ])

class_weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

final_model = build_final_model(X_train.shape[1])
final_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    ModelCheckpoint(str(MODEL_OUT), monitor="val_loss", save_best_only=True),
]

history = final_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=0,
)
print(f"Training stopped after {len(history.history['loss'])} epochs.")
print(f"Final model saved -> {MODEL_OUT}")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training stopped after 60 epochs.
Final model saved -> models/final_model.keras


## 4. Final Test-Set Evaluation

In [5]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

proba = final_model.predict(X_test, verbose=0).ravel()
pred = (proba >= FINAL_THRESHOLD).astype(int)

print(f"Final model @ threshold={FINAL_THRESHOLD}")
print(f"  Precision: {precision_score(y_test, pred, zero_division=0):.4f}")
print(f"  Recall:    {recall_score(y_test, pred, zero_division=0):.4f}")
print(f"  F1:        {f1_score(y_test, pred, zero_division=0):.4f}")
print(f"  AUC-PR:    {average_precision_score(y_test, proba):.4f}")


Final model @ threshold=0.93
  Precision: 1.0000
  Recall:    1.0000
  F1:        1.0000
  AUC-PR:    1.0000


## 📝 Summary

This notebook is the single source of truth for reproducing the final model: raw `creditcard.csv` in, `models/final_model.keras` + `models/final_scaler.joblib` out. The deployment app (`03_deployment/`) loads these two artifacts directly rather than repeating any of this logic.

**Next:** `03_deployment/` — a Streamlit app serving this exact model.
